# Análise da Razão de Escolha da Instituição (QE_I26)

**Pergunta:** *Qual a principal razão para você ter escolhido sua instituição de educação superior?*

| Código | Razão |
|--------|-------|
| A | Gratuidade |
| B | Preço da mensalidade |
| C | Proximidade da residência |
| D | Proximidade do trabalho |
| E | Facilidade de acesso |
| F | Qualidade/reputação |
| G | Única com aprovação |
| H | Possibilidade de bolsa |
| I | Outro motivo |

**Fonte dos dados:**
- `microdados2023_arq3.txt` → `NT_GER` (nota geral por aluno)
- `microdados2023_arq32.txt` → `QE_I26` (razão de escolha da instituição por aluno)

**Limitação:** Os microdados do ENADE não têm identificador de aluno — só `CO_CURSO`. Por isso não é possível ligar diretamente a nota de um aluno à sua razão de escolha. A estratégia adotada é:
1. Contar respostas brutas por categoria (visão dos alunos)
2. Para cada curso com >= 100 respostas, identificar a razão mais frequente (moda) e usar a nota média daquele curso
3. Relacionar nota média do curso com a razão predominante do curso

In [519]:
import polars as pl
import altair as alt
import numpy as np

alt.data_transformers.disable_max_rows()

DATA_DIR = r"C:\Users\ralex\Documents\GitHub\visualizacao-lab3\DADOS"

RAZOES = {
    "A": "Gratuidade",
    "B": "Preço da mensalidade",
    "C": "Proximidade da residência",
    "D": "Proximidade do trabalho",
    "E": "Facilidade de acesso",
    "F": "Qualidade/ reputação",
    "G": "Única com aprovação",
    "H": "Possibilidade de bolsa",
    "I": "Outro motivo",
}

In [520]:
arq3 = pl.read_csv(
    f"{DATA_DIR}/microdados2023_arq3.txt",
    separator=";", encoding="latin1",
    columns=["CO_CURSO", "NT_GER"],
    null_values=["", " "],
    infer_schema=False,
).with_columns([
    pl.col("CO_CURSO").cast(pl.Int64),
    pl.col("NT_GER").str.replace(",", ".").cast(pl.Float64, strict=False),
])

arq32 = pl.read_csv(
    f"{DATA_DIR}/microdados2023_arq32.txt",
    separator=";", encoding="latin1",
    columns=["CO_CURSO", "QE_I26"],
    null_values=["", " "],
).with_columns(pl.col("CO_CURSO").cast(pl.Int64))

media_por_curso = (
    arq3
    .filter(pl.col("NT_GER").is_not_null() & (pl.col("NT_GER") > 0))
    .group_by("CO_CURSO")
    .agg(pl.col("NT_GER").mean().alias("NT_GER_media"))
)

respostas = (
    arq32
    .filter(pl.col("QE_I26").is_not_null())
    .with_columns(pl.col("QE_I26").replace(RAZOES).alias("Razao"))
)

print(f"Alunos com resposta QE_I26: {respostas.shape[0]:,}")
print(f"Cursos com nota média:       {media_por_curso.shape[0]:,}")

Alunos com resposta QE_I26: 367,718
Cursos com nota média:       9,380


## 1. Distribuição bruta das respostas
Quantos alunos marcaram cada razão?

In [521]:
contagem_bruta = (
    respostas
    .group_by("Razao")
    .agg(pl.len().alias("n_respostas"))
    .sort("n_respostas", descending=True)
    .to_pandas()
)

total = contagem_bruta["n_respostas"].sum()
contagem_bruta["pct"] = (contagem_bruta["n_respostas"] / total * 100).round(1)

print(f"Total de respostas: {total:,}\n")
print(contagem_bruta.to_string(index=False))

Total de respostas: 367,718

                    Razao  n_respostas  pct
     Qualidade/ reputação       124113 33.8
Proximidade da residência        64051 17.4
     Preço da mensalidade        42169 11.5
               Gratuidade        41824 11.4
             Outro motivo        30176  8.2
   Possibilidade de bolsa        30096  8.2
     Facilidade de acesso        21677  5.9
      Única com aprovação         8807  2.4
  Proximidade do trabalho         4805  1.3


In [522]:
ordem_bruta = contagem_bruta.sort_values("n_respostas", ascending=False)["Razao"].tolist()
if "Outro motivo" in ordem_bruta:
    ordem_bruta.remove("Outro motivo")
    ordem_bruta.append("Outro motivo")

bar_respostas = (
    alt.Chart(contagem_bruta)
    .mark_bar()
    .encode(
        x=alt.X("Razao:N", sort=ordem_bruta, title=None,
                axis=alt.Axis(labelAngle=-30, labelLimit=180)),
        y=alt.Y("n_respostas:Q", title="Número de respostas"),
        color=alt.Color("Razao:N", legend=None),
        tooltip=["Razao", "n_respostas", "pct"],
    )
    .properties(title="Respostas brutas por razão (visão dos alunos)", width=580, height=360)
)

text_respostas = bar_respostas.mark_text(dy=-6, fontSize=11).encode(
    text=alt.Text("pct:Q", format=".1f"),
    color=alt.value("black"),
)

(bar_respostas + text_respostas)

alt.LayerChart(...)

## 2. Razão predominante por curso
Para cada curso com >= 100 respostas, qual é a razão mais citada pelos seus alunos (moda)?
Quantos cursos têm cada razão como predominante?

In [523]:
MIN_RESPOSTAS = 1

cursos_validos = (
    respostas
    .group_by("CO_CURSO")
    .agg(pl.len().alias("n_total"))
    .filter(pl.col("n_total") >= MIN_RESPOSTAS)
)

print(f"Cursos com >= {MIN_RESPOSTAS} respostas: {cursos_validos.shape[0]:,} de {respostas['CO_CURSO'].n_unique():,} no total")

moda_por_curso = (
    respostas
    .join(cursos_validos.select("CO_CURSO"), on="CO_CURSO", how="inner")
    .group_by(["CO_CURSO", "Razao"])
    .agg(pl.len().alias("n"))
    .group_by("CO_CURSO")
    .agg(
        pl.col("Razao")
        .sort_by(["n", "Razao"], descending=[True, False])
        .first()
    )
)

contagem_cursos = (
    moda_por_curso
    .group_by("Razao")
    .agg(pl.len().alias("n_cursos"))
    .sort("n_cursos", descending=True)
    .to_pandas()
)

total_cursos = contagem_cursos["n_cursos"].sum()
contagem_cursos["pct"] = (contagem_cursos["n_cursos"] / total_cursos * 100).round(1)

print(f"\nCursos por razão predominante:")
print(contagem_cursos.to_string(index=False))

Cursos com >= 1 respostas: 9,461 de 9,461 no total

Cursos por razão predominante:
                    Razao  n_cursos  pct
     Qualidade/ reputação      3489 36.9
Proximidade da residência      2273 24.0
               Gratuidade      1196 12.6
     Preço da mensalidade      1187 12.5
   Possibilidade de bolsa       713  7.5
             Outro motivo       384  4.1
     Facilidade de acesso       194  2.1
      Única com aprovação        19  0.2
  Proximidade do trabalho         6  0.1


In [524]:
ordem_cursos = contagem_cursos.sort_values("n_cursos", ascending=False)["Razao"].tolist()
if "Outro motivo" in ordem_cursos:
    ordem_cursos.remove("Outro motivo")
    ordem_cursos.append("Outro motivo")

bar_cursos = (
    alt.Chart(contagem_cursos)
    .mark_bar()
    .encode(
        x=alt.X("Razao:N", sort=ordem_cursos, title=None,
                axis=alt.Axis(labelAngle=-30, labelLimit=180)),
        y=alt.Y("n_cursos:Q", title="Número de cursos"),
        color=alt.Color("Razao:N", legend=None),
        tooltip=["Razao", "n_cursos", "pct"],
    )
    .properties(title="Cursos por razão predominante (moda dos alunos do curso)", width=580, height=360)
)

text_cursos = bar_cursos.mark_text(dy=-6, fontSize=11).encode(
    text=alt.Text("pct:Q", format=".1f"),
    color=alt.value("black"),
)

(bar_cursos + text_cursos)

alt.LayerChart(...)

## 3. Nota média × Razão de escolha da instituição

Cada ponto é um curso, classificado pela razão mais citada pelos seus alunos (moda).
A barra horizontal indica a **mediana** de cada grupo. Ordenado da maior para a menor mediana.

In [527]:
import pandas as pd

_df_strip = (
    moda_por_curso
    .join(media_por_curso, on="CO_CURSO", how="inner")
    .filter(pl.col("NT_GER_media").is_not_null())
    .select(["CO_CURSO", "Razao", "NT_GER_media"])
    .sort("CO_CURSO")
    .to_pandas()
)

_df_median = (
    _df_strip.groupby("Razao")["NT_GER_media"]
    .median()
    .reset_index()
    .rename(columns={"NT_GER_media": "mediana"})
)

ordem_nota = (
    _df_median.sort_values("mediana", ascending=False)["Razao"].tolist()
)
if "Outro motivo" in ordem_nota:
    ordem_nota.remove("Outro motivo")
    ordem_nota.append("Outro motivo")

rng = np.random.default_rng(100)
_df_strip["jitter"] = rng.uniform(-0.45, 0.45, size=len(_df_strip))

strip = (
    alt.Chart(_df_strip)
    .mark_circle(opacity=0.35, size=8)
    .encode(
        x=alt.X("Razao:N", sort=ordem_nota, title=None,
                axis=alt.Axis(
                    labelAngle=0,
                    labelLimit=100,
                    labelLineHeight=12,
                    labelExpr="split(datum.label, ' ')",
                )),
        xOffset=alt.XOffset("jitter:Q", scale=alt.Scale(domain=[-0.5, 0.5])),
        y=alt.Y("NT_GER_media:Q", title="Nota média do curso",
                scale=alt.Scale(zero=True)),
        color=alt.Color("Razao:N", legend=None),
        tooltip=["Razao", alt.Tooltip("NT_GER_media:Q", format=".1f", title="Nota")],
    )
)

mediana_layer = (
    alt.Chart(_df_median)
    .mark_tick(thickness=1, size=40, color="black", opacity=0.85)
    .encode(
        x=alt.X("Razao:N", sort=ordem_nota),
        y=alt.Y("mediana:Q"),
        tooltip=["Razao", alt.Tooltip("mediana:Q", format=".1f", title="Mediana")],
    )
)

label_layer = (
    alt.Chart(_df_median)
    .mark_text(dx=25, fontSize=11, fontWeight="bold", color="black", align="left", baseline="middle")
    .encode(
        x=alt.X("Razao:N", sort=ordem_nota),
        y=alt.Y("mediana:Q"),
        text=alt.Text("mediana:Q", format=".1f"),
    )
)

(strip + mediana_layer + label_layer).properties(
    title=alt.TitleParams(
        "Nota média do curso (NT_GER) × Razão predominante de escolha da instituição (QE_I26)",
    ),
    width=680, height=300,
)

alt.LayerChart(...)